In [1]:
from flask import Flask, render_template, request
from markupsafe import Markup
import numpy as np
import pandas as pd
import joblib
import requests
import config
import threading
from utils.fer import fertilizer_dic

In [2]:
import torch
import nest_asyncio
nest_asyncio.apply()

import cv2
import numpy as np
from PIL import Image
import io
from torchvision import transforms
#from flask import Flask, request, render_template_string, Markup, redirect


# Device setup (CPU in your case)
device = torch.device("cpu")
print("Using device:", device)


Using device: cpu


In [3]:
from flask import Flask, render_template, request
from markupsafe import Markup
import numpy as np
import pandas as pd
from utils.disease import disease_dic
from utils.fer import fertilizer_dic
import requests
import config
import pickle
import io
import torch
from torchvision import transforms
from PIL import Image
from utils.model import ResNet9
import torch
import torchvision.transforms as transforms
from PIL import Image
import torch.nn as nn

In [4]:
import torch
from utils.model import ResNet9  # Make sure ResNet9 is correctly imported

# Add your custom class to safe globals
torch.serialization.add_safe_globals([ResNet9])

disease_classes = [
    'Apple___Apple_scab',
    'Apple___Black_rot',
    'Apple___Cedar_apple_rust',
    'Apple___healthy',
    'Blueberry___healthy',
    'Cherry_(including_sour)___healthy',
    'Cherry_(including_sour)___Powdery_mildew',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_',
    'Corn_(maize)___healthy',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)',
    'Grape___healthy',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot',
    'Peach___healthy',
    'Pepper,_bell___Bacterial_spot',
    'Pepper,_bell___healthy',
    'Potato___Early_blight',
    'Potato___healthy',
    'Potato___Late_blight',
    'Raspberry___healthy',
    'Soybean___healthy',
    'Squash___Powdery_mildew',
    'Strawberry___healthy',
    'Strawberry___Leaf_scorch',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___healthy',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus'
]

# Set device (CPU in your case)
device = torch.device("cpu")

# Define the model file path
model_path = 'models/plant-disease-model-complete.pth'

# Attempt to load the full model or state_dict
try:
    disease_model = torch.load(model_path, map_location=device, weights_only=False)
except Exception as e:
    print("Full model load failed, trying state_dict:", e)
    disease_model = ResNet9(3, len(disease_classes))
    state_dict = torch.load(model_path, map_location=device, weights_only=False)
    disease_model.load_state_dict(state_dict)

disease_model.to(device)
disease_model.eval()
print("Plant disease model loaded successfully on", device)


Plant disease model loaded successfully on cpu


In [5]:
crop_recommendation_model_path = 'models/RandomForest.joblib'
crop_recommendation_model = joblib.load(open(crop_recommendation_model_path, 'rb'))

In [6]:
def weather_fetch(city_name):
    """Fetch weather data from OpenWeather API with error handling."""
    try:
        api_key = config.weather_api_key
        base_url = "http://api.openweathermap.org/data/2.5/weather?"
        complete_url = f"{base_url}appid={api_key}&q={city_name}"

        print(f"Fetching weather for: {city_name}")  # Debugging

        response = requests.get(complete_url, timeout=5)  # Timeout after 5 sec
        response.raise_for_status()  # Raise error for bad response codes (4xx, 5xx)
        
        data = response.json()
        print("API Response:", data)  # Debugging
        
        if data.get("cod") != 200:  # OpenWeather uses 200 for success
            print(f"Error: {data.get('message', 'Unknown error')}")
            return None
        
        # Extract temperature & humidity safely
        if "main" in data:
            temperature = round((data["main"]["temp"] - 273.15), 2)  # Convert to Celsius
            humidity = data["main"]["humidity"]
            return temperature, humidity
        else:
            print("Error: 'main' data not found in response.")
            return None

    except requests.exceptions.Timeout:
        print("Error: Weather API request timed out.")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error: Failed to fetch weather data. {e}")
        return None


In [7]:
def detect_leaf(img_bytes):
    """
    Returns True if a significant portion of the image is green (indicative of a leaf),
    otherwise returns False.
    """
    image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    image_np = np.array(image)
    hsv = cv2.cvtColor(image_np, cv2.COLOR_RGB2HSV)
    lower_green = np.array([25, 40, 40])
    upper_green = np.array([85, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    green_ratio = np.sum(mask > 0) / mask.size
    print("Green ratio:", green_ratio)  # Debug print
    return green_ratio > 0.15


In [8]:
def predict_disease(img_bytes, model, device):
    # Check if image is likely a leaf; if not, return an error message
    if not detect_leaf(img_bytes):
        return "Invalid input: Please upload an image of a plant leaf, not a face or non-leaf object."
    
    # Preprocess image (must match training preprocessing)
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.ToTensor(),
        # If normalization was applied during training, include it here.
        # Example: transforms.Normalize(mean=[0.485, 0.456, 0.406],
        #                                std=[0.229, 0.224, 0.225])
    ])
    image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    img_t = transform(image)
    img_u = torch.unsqueeze(img_t, 0).to(device)
    
    # Get raw model outputs
    outputs = model(img_u)
    print("Raw model outputs:", outputs)
    
    # Get predicted index and map to disease class
    _, preds = torch.max(outputs, dim=1)
    predicted_index = preds[0].item()
    print("Predicted index:", predicted_index)
    
    predicted_disease = disease_classes[predicted_index]
    print("Mapped Disease:", predicted_disease)
    
    # Optionally, use disease_dic to fetch a detailed description if available.
    if predicted_disease in disease_dic:
        return disease_dic[predicted_disease]
    else:
        return predicted_disease


In [9]:
app = Flask(__name__)

In [10]:
@app.route('/')
def home():
    title = 'Cropsure - Home'
    return render_template('index.html', title=title)

In [11]:
@app.route('/next')
def next_page():
    title = 'Cropsure - Next'
    return render_template('next.html', title=title)

In [12]:
@app.route('/crop-recommend')
def crop_recommend():
    title = 'Cropsure - Crop Recommendation'
    return render_template('crop.html', title=title)

In [13]:
@app.route('/fertilizer')
def fertilizer_recommendation():
    title = 'Cropsure - Fertilizer Suggestion'
    return render_template('fert.html', title=title)

In [14]:
@app.route('/plant-disease')
def plant_disease_detection():
    title = 'Cropsure - Plant Disease Detection'
    return render_template('plant disease.html', title=title)

In [15]:
@app.route('/try-again')
def try_again():
    title = 'Cropsure - Try Again'
    return render_template('try_again.html', title=title)

In [16]:
@app.route('/crop-predict', methods=['POST'])
def crop_prediction():
    
    if request.method == 'POST':
        try:
            N = request.form.get('nitrogen', type=float)
            P = request.form.get('phosphorus', type=float)
            K = request.form.get('potassium', type=float)
            ph = request.form.get('ph', type=float)
            rainfall = request.form.get('rainfall', type=float)
            city = request.form.get("city")

            print(f"Received values: N={N}, P={P}, K={K}, pH={ph}, Rainfall={rainfall}, City={city}")

            # Check if city is empty
            if not city:
                return "Error: City is required!", 400

            weather_data = weather_fetch(city)
            if weather_data:
                temperature, humidity = weather_data
                data = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
                print(f"Input to model: {data}")

                prediction = crop_recommendation_model.predict(data)[0]
                return render_template('crop result.html', prediction=prediction, title="Cropsure - Result")
            else:
                return render_template('try_again.html', title="Cropsure - Try Again")

        except Exception as e:
            print(f"Error occurred: {e}")  
            return f"Error: {e}"


In [17]:
@app.route('/fertilizer-predict', methods=['POST'])
def fert_recommend():
    try:
        title = 'Cropsure - Fertilizer Suggestion'
        crop_name = request.form.get('crop', '').strip()
        N = request.form.get('nitrogen', type=float)
        P = request.form.get('phosphorus', type=float)
        K = request.form.get('potassium', type=float)
        if not crop_name or N is None or P is None or K is None:
            return "Invalid input data. Please go back and fill all fields correctly.", 400

        try:
            df = pd.read_csv('Data/fertilizer.csv')
        except FileNotFoundError:
            return "Fertilizer data file not found. Check file path.", 500

        if crop_name not in df['Crop'].values:
            return f"Crop '{crop_name}' not found in database.", 400

        nr, pr, kr = df[df['Crop'] == crop_name][['N', 'P', 'K']].iloc[0]
        
        n, p, k = nr - N, pr - P, kr - K
        temp = {abs(n): "N", abs(p): "P", abs(k): "K"}
        max_value = temp[max(temp.keys())]

        key = f"{max_value}{'High' if eval(max_value.lower()) < 0 else 'low'}"
        recommendation = fertilizer_dic.get(key, "No recommendation found.")

        return render_template('fertilizer result.html', recommendation=Markup(recommendation), title=title)
    
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}", 500



In [18]:
@app.route('/disease-predict', methods=['GET', 'POST'])
def disease_prediction():
    title = 'Harvestify - Disease Detection'
    if request.method == 'POST':
        if 'file' not in request.files:
            return redirect(request.url)
        file = request.files.get('file')
        if not file:
            return render_template('plant disease.html', title=title)
        try:
            img = file.read()
            prediction = predict_disease(img, disease_model, device)
            prediction = Markup(str(prediction))
            return render_template('plant result.html', prediction=prediction, title=title)
        except Exception as e:
            return f"An error occurred: {e}"
    return render_template('plant disease.html', title=title)


In [ ]:
from flask import Flask
from werkzeug.serving import run_simple

if __name__ == '__main__':
    run_simple('localhost', 5000, app)




 * Running on http://localhost:5000
Press CTRL+C to quit
127.0.0.1 - - [02/Apr/2025 12:31:57] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2025 12:31:58] "GET /static/images/crop%20reccomen.jpeg HTTP/1.1" 304 -
